In [2]:
!git clone https://github.com/gdmarmerola/cfml_tools.git
import sys
import os

import sys
import os

# Em vez de adicionar base_dir, vamos adicionar o caminho direto onde o pacote vive
repo_path = os.path.join(os.getcwd(), 'cfml_tools')
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Tenta importar novamente
from cfml_tools.tree import DecisionTreeCounterfactual
print("Importação bem-sucedida!")

fatal: destination path 'cfml_tools' already exists and is not an empty directory.
Importação bem-sucedida!


In [3]:
import sys
import os
import gc
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ============================================
# 0. SETUP E IMPORTAÇÃO
# ============================================
base_dir = os.getcwd()
repo_dir = os.path.join(base_dir, 'cfml_tools')
if not os.path.exists(repo_dir):
    os.system(f"git clone https://github.com/gdmarmerola/cfml_tools.git {repo_dir}")
if base_dir not in sys.path: sys.path.insert(0, base_dir)

from cfml_tools.tree import DecisionTreeCounterfactual
warnings.filterwarnings('ignore')

# ============================================
# 1. CONFIGURAÇÃO
# ============================================
TARGETS = ['ABE_ESP', 'ABC_ESP', 'ABB_ESP', 'ABI_ESP']
TREATMENT = 'ADITIVO'
COVARS = ['REGIAO', 'ESPÉCIE', 'FASE', 'AREA', 'ABH_ESP']
df = pd.read_csv('/usr/app/Embrapa/embrapa.csv')

# ============================================
# 2. PREPARAÇÃO E PIPELINE
# ============================================
COVARS_CAT = [c for c in COVARS if df[c].dtype == 'object' or df[c].nunique() <= 20]
COVARS_NUM = [c for c in COVARS if c not in COVARS_CAT]

df[TREATMENT] = df[TREATMENT].astype('category')
W_full = df[TREATMENT].cat.codes.values

preprocessor = ColumnTransformer(transformers=[
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), COVARS_CAT),
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), COVARS_NUM)
])

# ============================================
# 3. PROPENSITY E OVERLAP
# ============================================
X_full = preprocessor.fit_transform(df[COVARS_CAT + COVARS_NUM])
ps_model = LogisticRegression(multi_class='multinomial', max_iter=500).fit(X_full, W_full)
mask_overlap = (ps_model.predict_proba(X_full) > 0.1).sum(axis=1) >= 2
df_overlap = df[mask_overlap].copy()
W_overlap = W_full[mask_overlap]
X_overlap = preprocessor.transform(df_overlap[COVARS_CAT + COVARS_NUM])

# ============================================
# 4. FUNÇÃO DE DIAGNÓSTICO E RODAGEM
# ============================================
def rodar_y(y_name):
    print(f"\n--- Diagnóstico para {y_name} ---")
    y_vals = df_overlap[y_name].fillna(df_overlap[y_name].median()).values
    
    feature_names = preprocessor.get_feature_names_out()
    X_df = pd.DataFrame(X_overlap, columns=feature_names)
    
    # Filtro reduzido para não descartar efeitos sutis
    dtcf = DecisionTreeCounterfactual(min_sample_effect=5, save_explanatory=True)
    dtcf.fit(X_df, W_overlap, y_vals)
    cf = pd.DataFrame(dtcf.predict(X_df), columns=[f'pred_W{i}' for i in range(len(np.unique(W_overlap)))])
    
    print("Médias das predições:", cf.mean().values)
    diff = cf.max(axis=1) - cf.min(axis=1)
    print(f"Diferença média entre melhor/pior aditivo: {diff.mean():.6f}")
    
    effects = cf.sub(cf['pred_W0'], axis=0)
    
    df_res = df_overlap[COVARS + [y_name]].copy()
    col_vencedora = effects.idxmax(axis=1)
    df_res['melhor_aditivo'] = pd.to_numeric(col_vencedora.apply(lambda x: str(x).replace('pred_W', '')), errors='coerce').fillna(0).astype(int)
    df_res['ganho_max'] = effects.max(axis=1).fillna(0)
    df_res['y'] = y_name
    return df_res

# Execução
all_effects = [rodar_y(y) for y in TARGETS]
df_final = pd.concat(all_effects, ignore_index=True)
df_final.to_csv('resultado_debug.csv', index=False)

print("\nProcessamento finalizado. Verifique a 'Diferença média' acima.")


--- Diagnóstico para ABE_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABC_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABB_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABI_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

Processamento finalizado. Verifique a 'Diferença média' acima.


In [10]:
import sys
import subprocess

# Instalar o statsmodels usando o gerenciador de pacotes pip
subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])

# Após a instalação, você pode importá-lo normalmente
import statsmodels.api as sm
print("Statsmodels instalado e importado com sucesso!")
import pandas as pd
import statsmodels.formula.api as smf

# 1. Definições
TARGETS = ['ABE_ESP', 'ABC_ESP', 'ABB_ESP', 'ABI_ESP']

from statsmodels.stats.multicomp import pairwise_tukeyhsd

from statsmodels.stats.multicomp import pairwise_tukeyhsd

def analisar_todas_relacoes(df, targets):
    todas_comparacoes = []
    
    for y in targets:
        print(f"\n--- Analisando {y} ---")
        
        # Realiza o teste
        try:
            tukey = pairwise_tukeyhsd(endog=df[y], groups=df['ADITIVO'], alpha=0.05)
            
            # Converte para DataFrame
            df_tukey = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
            
            # Filtra apenas os significativos
            df_sig = df_tukey[df_tukey['reject'] == True].copy()
            
            if df_sig.empty:
                print(f"Nenhuma diferença significativa encontrada para {y}.")
            else:
                print(f"Encontradas {len(df_sig)} relações significativas para {y}.")
                for _, row in df_sig.iterrows():
                    todas_comparacoes.append({
                        'Alvo (y)': y,
                        'Comparação': f"{row['group1']} vs {row['group2']}",
                        'Diferença Média': round(row['meandiff'], 4),
                        'P-Valor': round(row['p-adj'], 4),
                        'Melhor': row['group2'] if row['meandiff'] > 0 else row['group1']
                    })
        except Exception as e:
            print(f"Erro ao processar {y}: {e}")
            
    return pd.DataFrame(todas_comparacoes)

# Execução
relatorio_completo = analisar_todas_relacoes(df, TARGETS)

# Verificação
if not relatorio_completo.empty:
    print("\n--- RESUMO FINAL ---")
    print(relatorio_completo)
else:
    print("\nNenhuma relação significativa foi encontrada para nenhum dos alvos.")

Statsmodels instalado e importado com sucesso!

--- Analisando ABE_ESP ---
Encontradas 4 relações significativas para ABE_ESP.

--- Analisando ABC_ESP ---
Nenhuma diferença significativa encontrada para ABC_ESP.

--- Analisando ABB_ESP ---
Nenhuma diferença significativa encontrada para ABB_ESP.

--- Analisando ABI_ESP ---
Nenhuma diferença significativa encontrada para ABI_ESP.

--- RESUMO FINAL ---
  Alvo (y) Comparação  Diferença Média  P-Valor Melhor
0  ABE_ESP     B vs I        1473.8205   0.0000      I
1  ABE_ESP     C vs I        1218.0852   0.0001      I
2  ABE_ESP     D vs E        1046.7182   0.0476      E
3  ABE_ESP     D vs I        1636.5426   0.0000      I
